
# Post-Cleaning Exploratory Data Analysis (EDA)

## Amazon ML Challenge 2026 — Entity Resolution



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base_path = "/content/drive/MyDrive/Amazon_ML_Challenge_2026/cleaned_data"
print("Folder Exists: ", os.path.exists(base_path))
if os.path.exists(base_path):
  print("\nFiles in cleaned_data:")
  for file in os.listdir(base_path):
    print(file)

Folder Exists:  True

Files in cleaned_data:
train_source1_cleaned.tsv
test_source1_cleaned.tsv
test_source2_cleaned.tsv
train_source2_cleaned.tsv
test_source3_cleaned.tsv
train_source3_cleaned.tsv


In [ ]:
import pandas as pd

#path
s1_path = f"{base_path}/train_source1_cleaned.tsv"
s2_path = f"{base_path}/train_source2_cleaned.tsv"
s3_path = f"{base_path}/train_source3_cleaned.tsv"
#load dataset
s1 = pd.read_csv(s1_path, sep="\t", dtype=str)
s2 = pd.read_csv(s2_path, sep="\t", dtype=str)
s3 = pd.read_csv(s3_path, sep="\t", dtype=str)
print("All cleaned train datasets loaded successfully")


All cleaned train datasets loaded successfully


## Dataset Overview

In [ ]:
datasets = {
    "Source 1": s1,
    "Source 2": s2,
    "Source 3": s3
}
for name, df in datasets.items():
  print(f"\n{name}")
  print("-" * 50)
  print("Rows:", df.shape[0])
  print("Columns:", df.shape[1])
  print("Column names:", df.columns.tolist())


Source 1
--------------------------------------------------
Rows: 2206821
Columns: 8
Column names: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_clean', 'business_address_clean', 'business_name_normalized', 'business_address_normalized']

Source 2
--------------------------------------------------
Rows: 5034616
Columns: 8
Column names: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_clean', 'business_address_clean', 'business_name_normalized', 'business_address_normalized']

Source 3
--------------------------------------------------
Rows: 5285603
Columns: 8
Column names: ['entity_id', 'business_name', 'business_address', 'country', 'business_name_clean', 'business_address_clean', 'business_name_normalized', 'business_address_normalized']



## Missing and Empty Value Analysis



In [ ]:
columns_to_check = [
    "business_name_clean",
    "business_address_clean",
    "business_name_normalized",
    "business_address_normalized"
]
results = []
for source_name, df in datasets.items():
  for column in columns_to_check:
    values = df[column]
    missing_count = values.isna().sum()
    empty_count = (
        values.notna() & values.str.strip().eq("")
    ).sum()
    results.append({
        "Source": source_name,
        "Column": column,
        "Missing values": missing_count,
        "Empty values": empty_count
    })
summary_df = pd.DataFrame(results)
display(summary_df)

,Source,Column,Missing values,Empty values
0,Source 1,business_name_clean,0,0
1,Source 1,business_address_clean,0,0
2,Source 1,business_name_normalized,0,0
3,Source 1,business_address_normalized,0,0
4,Source 2,business_name_clean,2,0
5,Source 2,business_address_clean,168967,0
6,Source 2,business_name_normalized,3,0
7,Source 2,business_address_normalized,168967,0
8,Source 3,business_name_clean,13,0
9,Source 3,business_address_clean,175916,0


## Duplicate Analysis



In [ ]:
for source_name, df in datasets.items():
    duplicate_count = df.duplicated().sum()

    print(f"{source_name}:")
    print("Exact duplicate rows:", duplicate_count)
    print()

Source 1:
Exact duplicate rows: 0

Source 2:
Exact duplicate rows: 0

Source 3:
Exact duplicate rows: 0



In [ ]:
for source_name, df in datasets.items():
    duplicate_ids = df["entity_id"].duplicated().sum()

    print(f"{source_name}:")
    print("  Duplicate entity IDs:", duplicate_ids)
    print()

Source 1:
  Duplicate entity IDs: 0

Source 2:
  Duplicate entity IDs: 0

Source 3:
  Duplicate entity IDs: 0



In [ ]:
for source_name, df in datasets.items():
    repeated_name_count = (
        df["business_name_normalized"]
        .dropna()
        .duplicated()
        .sum()
    )

    print(f"{source_name}:")
    print("  Repeated normalized-name occurrences:", repeated_name_count)
    print()

Source 1:
  Repeated normalized-name occurrences: 686137

Source 2:
  Repeated normalized-name occurrences: 1006434

Source 3:
  Repeated normalized-name occurrences: 1003170



In [ ]:
for source_name, df in datasets.items():
    name_counts = (
        df["business_name_normalized"]
        .dropna()
        .value_counts()
    )

    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    display(
        name_counts.head(10).rename_axis(
            "Normalized Business Name"
        ).reset_index(name="Record Count")
    )


Source 1


,Normalized Business Name,Record Count
0,primary care group,253
1,ear nose throat group,251
2,pediatric group,222
3,womens health group,220
4,physical therapy group,218
5,pediatric dental group,216
6,behavioral health group,215
7,chiropractic group,209
8,orthopedic group,208
9,eye group,207



Source 2


,Normalized Business Name,Record Count
0,physical therapy,526
1,primary care,518
2,womens health,483
3,urgent care,469
4,pediatric dental,468
5,internal medicine,468
6,behavioral health,468
7,pediatric dentistry,453
8,foot ankle,369
9,ear nose throat,362



Source 3


,Normalized Business Name,Record Count
0,primary care,521
1,physical therapy,516
2,pediatric dental,503
3,urgent care,480
4,womens health,469
5,pediatric dentistry,465
6,internal medicine,427
7,behavioral health,427
8,foot ankle,355
9,pediatric,326


## Country Distribution

In [ ]:
for source_name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    display(
        df["country"]
        .value_counts(dropna=False)
        .rename_axis("Country")
        .reset_index(name="Record Count")
    )


Source 1


,Country,Record Count
0,US,1323633
1,India,883188



Source 2


,Country,Record Count
0,US,3016817
1,India,2017799



Source 3


,Country,Record Count
0,US,3170056
1,India,2115547


## Business Name Cleaning and Normalization

In [ ]:
for source_name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    display(
        df[
            [
                "business_name",
                "business_name_clean",
                "business_name_normalized"
            ]
        ].head(10)
    )


Source 1


,business_name,business_name_clean,business_name_normalized
0,Orelee's Barbershop,Orelee's Barbershop,orelee s barbershop
1,Prime Money,Prime Money,prime money
2,B+ Retail Inc,B+ Retail Inc,b retail inc
3,Christ Chapel,Christ Chapel,christ chapel
4,Prabhav Business Center,Prabhav Business Center,prabhav business center
5,Custom Wealth Services LLC,Custom Wealth Services LLC,custom wealth services llc
6,Consulting Nyasa Nursing Private Limited,Consulting Nyasa Nursing Private Limited,consulting nyasa nursing private limited
7,Nexus Anchor Rain,Nexus Anchor Rain,nexus anchor rain
8,Moore Bitwise Inc,Moore Bitwise Inc,moore bitwise inc
9,Dermatology Green Medicine,Dermatology Green Medicine,dermatology green medicine



Source 2


,business_name,business_name_clean,business_name_normalized
0,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग प्राइवेट लिमिटेड,राम मार्केटिंग प्राइवेट लिमिटेड
1,-- Holloway Peak Inc Seafood,-- Holloway Peak Inc Seafood,holloway peak inc seafood
2,आदित्य प्रॉपर्टीज एलएलपी,आदित्य प्रॉपर्टीज एलएलपी,आदित्य प्रॉपर्टीज एलएलपी
3,Summit Inc,Summit Inc,summit inc
4,Delta Tetlecommunication Inc,Delta Tetlecommunication Inc,delta tetlecommunication inc
5,Lee and Lawson,Lee and Lawson,lee and lawson
6,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,SHIVSHAKTI VIDYALAYA VIDYALAYA OVERSEAS CORPOR...,shivshakti vidyalaya vidyalaya overseas corpor...
7,Shree Infracon Private Ltd,Shree Infracon Private Ltd,shree infracon private ltd
8,Chavira Platinum Chimera LLC,Chavira Platinum Chimera LLC,chavira platinum chimera llc
9,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,FOUNDATION EXCEL AGENCY PRIVATE LIMITED,foundation excel agency private limited



Source 3


,business_name,business_name_clean,business_name_normalized
0,wilfordhancock.com,wilfordhancock.com,wilfordhancock com
1,International South Consultants Private Ltd,International South Consultants Private Ltd,international south consultants private ltd
2,LLC Moncada Léarning Center,LLC Moncada Léarning Center,llc moncada léarning center
3,Moyna's Coffee,Moyna's Coffee,moyna s coffee
4,Pvt. EFS Print Ventures Ltd.,Pvt. EFS Print Ventures Ltd.,pvt efs print ventures ltd
5,LLC Hernandez Colonial Redwood,LLC Hernandez Colonial Redwood,llc hernandez colonial redwood
6,Classic Equity Partners Group,Classic Equity Partners Group,classic equity partners group
7,Ectolumdrex dba X+ Madison Inc,Ectolumdrex dba X+ Madison Inc,ectolumdrex dba x madison inc
8,Animal Hanisch Hospirlg,Animal Hanisch Hospirlg,animal hanisch hospirlg
9,Gomez Optimal,Gomez Optimal,gomez optimal


## Business Address Cleaning and Normalization

In [ ]:
address_columns = [
    "business_address",
    "business_address_clean",
    "business_address_normalized"
]

for source_name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    display(df[address_columns].head(10))



Source 1


,business_address,business_address_clean,business_address_normalized
0,"1795 Westchester Drive, High Point, NC","1795 Westchester Drive, High Point, NC",1795 westchester drive high point nc
1,"17560 Ellis Road, Tahlequah, OK","17560 Ellis Road, Tahlequah, OK",17560 ellis road tahlequah ok
2,"1712 Montebello Avenue, Phoenix, AZ","1712 Montebello Avenue, Phoenix, AZ",1712 montebello avenue phoenix az
3,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD","2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",2100 cameron drive unit apartment g dundalk md
4,"797, Lake Town Block A, Kolkata, Howrah, West ...","797, Lake Town Block A, Kolkata, Howrah, West ...",797 lake town block a kolkata howrah west bengal
5,"OH, Columbus, 5559 Orville Avenue","OH, Columbus, 5559 Orville Avenue",oh columbus 5559 orville avenue
6,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...","2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",2505 tower 1 oakwood runwal greens mulund gore...
7,"1111 Church Street, Unit 2007, Nashville, TN","1111 Church Street, Unit 2007, Nashville, TN",1111 church street unit 2007 nashville tn
8,"337 Oakland Avenue, Michigan City, IN","337 Oakland Avenue, Michigan City, IN",337 oakland avenue michigan city in
9,"294 Meadowcreek Drive, Unit Unit 2, Village Of...","294 Meadowcreek Drive, Unit Unit 2, Village Of...",294 meadowcreek drive unit unit 2 village of p...



Source 2


,business_address,business_address_clean,business_address_normalized
0,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi","KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",kh no 570 13 new delhi west delhi delhi
1,"105 ELM ST, MORGANTON, NC","105 ELM ST, MORGANTON, NC",105 elm st morganton nc
2,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh","G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",g 3 571 gulmohar colony bhopal madhya pradesh
3,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL","GREENSBORO, NC, 19 1/2 STARDUST TRAIL",greensboro nc 19 1 2 stardust trail
4,"914 PIERPONT AVE, CLEVELAND, OH","914 PIERPONT AVE, CLEVELAND, OH",914 pierpont ave cleveland oh
5,"1702 Pine Avenue, CITY OF MENOMONIE, WI","1702 Pine Avenue, CITY OF MENOMONIE, WI",1702 pine avenue city of menomonie wi
6,"H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab","H.NO 204 C ROAD HOSHIARPUR, PUNJAB, Punjab",h no 204 c road hoshiarpur punjab punjab
7,"63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...","63/2275/7, ALHIND TOWER, FIRST FLOOR, JAFFERKH...",63 2275 7 alhind tower first floor jafferkhan ...
8,"282 SAXONY DRIVE, FTT MITCHELL, KY","282 SAXONY DRIVE, FTT MITCHELL, KY",282 saxony drive ftt mitchell ky
9,"HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...","HN 753 E-1, BHARAT NAGAR, 104/1/1 ERANDWANE, M...",hn 753 e 1 bharat nagar 104 1 1 erandwane maha...



Source 3


,business_address,business_address_clean,business_address_normalized
0,"Mack Rd, Haltom City, Texas","Mack Rd, Haltom City, Texas",mack rd haltom city texas
1,NaN,NaN,NaN
2,"5780 Fawn Ct, Fort Worth, Texas","5780 Fawn Ct, Fort Worth, Texas",5780 fawn ct fort worth texas
3,"1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio","1 Ivanhoe Ave, PO Box 6009, Cincinnati, Ohio",1 ivanhoe ave po box 6009 cincinnati ohio
4,"Door No 183, 41St Cross, 22Nd Main 9Th Block J...","Door No 183, 41St Cross, 22Nd Main 9Th Block J...",door no 183 41st cross 22nd main 9th block jay...
5,"2260- Housecreek Trail, Unit 407, Raleigh, Nor...","2260- Housecreek Trail, Unit 407, Raleigh, Nor...",2260 housecreek trail unit 407 raleigh north c...
6,"6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...","6885 Catalpa Bluff Ln, PO Box 5799, Dickinson,...",6885 catalpa bluff ln po box 5799 dickinson texas
7,"S03575 Cty Tk M, Town Of Buffalo, WI","S03575 Cty Tk M, Town Of Buffalo, WI",s03575 cty tk m town of buffalo wi
8,"##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri","##8 Willow Oak Lane, Fl. 0, Saint Louis, Missouri",8 willow oak lane fl 0 saint louis missouri
9,"343 Hempstead 161, Hope, Arkansas","343 Hempstead 161, Hope, Arkansas",343 hempstead 161 hope arkansas


In [ ]:
for source_name, df in datasets.items():
    original = df["business_address"]
    cleaned = df["business_address_clean"]

    unchanged = (
        original.eq(cleaned)
        | (original.isna() & cleaned.isna())
    )

    changed_count = (~unchanged).sum()

    print(f"{source_name}:")
    print("  Addresses changed during cleaning:", changed_count)
    print("  Total records:", len(df))
    print()

Source 1:
  Addresses changed during cleaning: 1572
  Total records: 2206821

Source 2:
  Addresses changed during cleaning: 108790
  Total records: 5034616

Source 3:
  Addresses changed during cleaning: 5945
  Total records: 5285603



### Address Changes During Cleaning

In [ ]:
for source_name, df in datasets.items():
    original = df["business_address"]
    cleaned = df["business_address_clean"]

    unchanged = (
        original.eq(cleaned)
        | (original.isna() & cleaned.isna())
    )

    changed_rows = df.loc[
        ~unchanged,
        [
            "business_address",
            "business_address_clean",
            "business_address_normalized"
        ]
    ]

    print(f"\n{'=' * 50}")
    print(source_name)
    print("Changed address examples")
    print("=" * 50)

    display(changed_rows.head(10))


Source 1
Changed address examples


,business_address,business_address_clean,business_address_normalized
862,"A-212, Malhotra Complex, Gali No. Â 01, Vika...","A-212, Malhotra Complex, Gali No. – 01, Vikas ...",a 212 malhotra complex gali no 01 vikas marg s...
2140,"5320 81st Street, Unit APARTMENT 733, Tulsa, OK","5320 81st Street, Unit APARTMENT 733, Tulsa, OK",5320 81st street unit apartment 733 tulsa ok
4668,"4886 Darlington Avenue, Unit APARTMENT 315, T...","4886 Darlington Avenue, Unit APARTMENT 315, Tu...",4886 darlington avenue unit apartment 315 tuls...
7514,"7802 97th Street, Unit APARTMENT 7201, Tulsa, OK","7802 97th Street, Unit APARTMENT 7201, Tulsa, OK",7802 97th street unit apartment 7201 tulsa ok
7585,"3658 Maplewood Avenue, Unit APARTMENT C, Tuls...","3658 Maplewood Avenue, Unit APARTMENT C, Tulsa...",3658 maplewood avenue unit apartment c tulsa ok
7831,"13761 51st Street, Unit APARTMENT 5204, Tulsa...","13761 51st Street, Unit APARTMENT 5204, Tulsa, OK",13761 51st street unit apartment 5204 tulsa ok
11322,"619 11th Street, Unit APARTMENT 6, Tulsa, OK","619 11th Street, Unit APARTMENT 6, Tulsa, OK",619 11th street unit apartment 6 tulsa ok
12621,"C 12/27 ÂA & C 12/28, Lahangpura Aurangabad,...","C 12/27 –A & C 12/28, Lahangpura Aurangabad, V...",c 12 27 a c 12 28 lahangpura aurangabad varana...
13370,"8342 25th Place, Unit APARTMENT 131, Tulsa, OK","8342 25th Place, Unit APARTMENT 131, Tulsa, OK",8342 25th place unit apartment 131 tulsa ok
13724,"7311 99th Avenue E, Unit APARTMENT 1320, Tuls...","7311 99th Avenue E, Unit APARTMENT 1320, Tulsa...",7311 99th avenue e unit apartment 1320 tulsa ok



Source 2
Changed address examples


,business_address,business_address_clean,business_address_normalized
38,"1225 GUMWOOD DRIVE, OH, PLEASANT TWP","1225 GUMWOOD DRIVE, OH, PLEASANT TWP",1225 gumwood drive oh pleasant twp
39,"2062 SMITH HOLLOW RD, BLAINE, TN","2062 SMITH HOLLOW RD, BLAINE, TN",2062 smith hollow rd blaine tn
78,"4314 COBBLEWOOD CT, INDEPENDENCE, KY","4314 COBBLEWOOD CT, INDEPENDENCE, KY",4314 cobblewood ct independence ky
122,"6535 PREMIER DR, NASHVLLE CDP, TN","6535 PREMIER DR, NASHVLLE CDP, TN",6535 premier dr nashvlle cdp tn
151,"Telangana, H.NO 39 II FLOOR NEAR JASMINE HIGH...","Telangana, H.NO 39 II FLOOR NEAR JASMINE HIGH ...",telangana h no 39 ii floor near jasmine high s...
202,"256 ROSELAND PLACE, ABERDEEN, NC","256 ROSELAND PLACE, ABERDEEN, NC",256 roseland place aberdeen nc
203,"106 KENNEDY ROAD, LOUDOUN COUNTY, VA","106 KENNEDY ROAD, LOUDOUN COUNTY, VA",106 kennedy road loudoun county va
208,MANISH MARKET G-68 SENAPATIBAPAT MARG DADAR W...,MANISH MARKET G-68 SENAPATIBAPAT MARG DADAR WE...,manish market g 68 senapatibapat marg dadar we...
216,"162 SOUTH AVENUE, WI, VILLAGE OF CLEAR LAKE","162 SOUTH AVENUE, WI, VILLAGE OF CLEAR LAKE",162 south avenue wi village of clear lake
267,"4117 CORNELIUS AVE, PMB 5858, INDIANAPOLIS, IN","4117 CORNELIUS AVE, PMB 5858, INDIANAPOLIS, IN",4117 cornelius ave pmb 5858 indianapolis in



Source 3
Changed address examples


,business_address,business_address_clean,business_address_normalized
135,"New York, Highland Falls, 282 Main St","New York, Highland Falls, 282 Main St",new york highland falls 282 main st
330,"1363 84 St, Brooklyn, New York","1363 84 St, Brooklyn, New York",1363 84 st brooklyn new york
1786,"Plot 416 Shop Â 484, Mumbai, Mumbai City, MH","Plot 416 Shop – 484, Mumbai, Mumbai City, MH",plot 416 shop 484 mumbai mumbai city mh
2217,"N13308 N 10nd Avenue, Hurley CITY, Wisconsin","N13308 N 10nd Avenue, Hurley CITY, Wisconsin",n13308 n 10nd avenue hurley city wisconsin
2770,"17566 111 Avenue, Jamaica, New York","17566 111 Avenue, Jamaica, New York",17566 111 avenue jamaica new york
3787,"H 316 Snowpine Village, Great Valley, New York","H 316 Snowpine Village, Great Valley, New York",h 316 snowpine village great valley new york
4173,"Vayu Â 616, Sy.no 79, Ele Room Â I, Near H...","Vayu – 616, Sy.no 79, Ele Room – I, Near Hitec...",vayu 616 sy no 79 ele room i near hitech city ...
4590,"2211 Yeoman St, Waukegan, Illinois","2211 Yeoman St, Waukegan, Illinois",2211 yeoman st waukegan illinois
4864,"135 Antlers Ln, Bear, Delaware","135 Antlers Ln, Bear, Delaware",135 antlers ln bear delaware
5073,"4688 Glendon Rd, Ohio, Columbus","4688 Glendon Rd, Ohio, Columbus",4688 glendon rd ohio columbus


## Repeated Normalized Address Analysis

In [ ]:
for source_name, df in datasets.items():
    address_counts = (
        df["business_address_normalized"]
        .dropna()
        .value_counts()
    )

    repeated_occurrences = (
        address_counts[address_counts > 1] - 1
    ).sum()

    print(f"{source_name}:")
    print("  Repeated address occurrences:", repeated_occurrences)
    print("  Distinct normalized addresses:", len(address_counts))
    print()

Source 1:
  Repeated address occurrences: 76653
  Distinct normalized addresses: 2130168

Source 2:
  Repeated address occurrences: 579575
  Distinct normalized addresses: 4286074

Source 3:
  Repeated address occurrences: 493646
  Distinct normalized addresses: 4616041



### Most Frequent Normalized Addresses

In [ ]:
for source_name, df in datasets.items():
    address_counts = (
        df["business_address_normalized"]
        .dropna()
        .value_counts()
    )

    print(f"\n{'=' * 50}")
    print(source_name)
    print("Top 10 normalized addresses")
    print("=" * 50)

    display(
        address_counts.head(10)
        .rename_axis("Normalized Business Address")
        .reset_index(name="Record Count")
    )


Source 1
Top 10 normalized addresses


,Normalized Business Address,Record Count
0,108 norle street college twp pa,14
1,104 roadrunner circle elephant butte nm,13
2,1921 willow park drive fort worth tx,13
3,43 franklin street delaware oh,13
4,218 phillips street storm lake ia,13
5,55 lavkush vihar singhpur kachhar bithoor road...,13
6,h no 154 a veer savarkar nagar mini bypass roa...,12
7,no 109 ferns meadows avalahalli main road bile...,12
8,333 court avenue selmer tn,12
9,4804 gardenia avenue glendale az,12



Source 2
Top 10 normalized addresses


,Normalized Business Address,Record Count
0,1948 weaver forest way morrisville nc,19
1,17200 bell rd surprise az,19
2,3710 goldfield rd apache junction az,18
3,842 38 caledonai township wi,18
4,66 frank edge alto tx,17
5,908 power road mesa desert az,16
6,1110 henness road casa grande az,16
7,06252 taliaferro way aleandria va,16
8,1105 m null birmingham al,16
9,g 58 1 benson cross road bangalore karnataka,16



Source 3
Top 10 normalized addresses


,Normalized Business Address,Record Count
0,no 1 chennai tn,35
1,1 bangalore ka,35
2,18 kolkata howrah wb,32
3,1 kolkata howrah wb,32
4,no 5 bangalore ka,32
5,no 7 bangalore ka,30
6,ground floor bangalore ka,29
7,no 3 bangalore ka,29
8,5 ahmedabad gj,28
9,no 2 chennai tn,28


##Name and Address Availability

In [ ]:
for source_name, df in datasets.items():
    name_available = (
        df["business_name_normalized"].notna()
        & df["business_name_normalized"].str.strip().ne("")
    )

    address_available = (
        df["business_address_normalized"].notna()
        & df["business_address_normalized"].str.strip().ne("")
    )

    both_available = name_available & address_available
    name_only = name_available & ~address_available
    address_only = ~name_available & address_available
    neither_available = ~name_available & ~address_available

    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)
    print("Both available:", both_available.sum())
    print("Name only:", name_only.sum())
    print("Address only:", address_only.sum())
    print("Neither available:", neither_available.sum())


Source 1
Both available: 2206821
Name only: 0
Address only: 0
Neither available: 0

Source 2
Both available: 4865646
Name only: 168967
Address only: 3
Neither available: 0

Source 3
Both available: 5109674
Name only: 175916
Address only: 13
Neither available: 0


##Business Name Length Analysis

In [ ]:
for source_name, df in datasets.items():
    name_lengths = (
        df["business_name_normalized"]
        .dropna()
        .str.len()
    )

    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    print("Minimum length:", name_lengths.min())
    print("Average length:", round(name_lengths.mean(), 2))
    print("Median length:", name_lengths.median())
    print("Maximum length:", name_lengths.max())


Source 1
Minimum length: 3
Average length: 23.7
Median length: 24.0
Maximum length: 105

Source 2
Minimum length: 1
Average length: 24.48
Median length: 24.0
Maximum length: 102

Source 3
Minimum length: 1
Average length: 24.58
Median length: 24.0
Maximum length: 123


### Short Business Names

In [ ]:
for source_name, df in datasets.items():
    names = df["business_name_normalized"].dropna().str.strip()

    short_names = names[names.str.len() <= 5]

    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    print("Names with 5 or fewer characters:", len(short_names))

    display(
        short_names.value_counts()
        .head(10)
        .rename_axis("Normalized Business Name")
        .reset_index(name="Record Count")
    )


Source 1
Names with 5 or fewer characters: 6848


,Normalized Business Name,Record Count
0,lynx,102
1,birch,98
2,forge,96
3,amber,96
4,nova,93
5,aspen,91
6,cedar,89
7,ember,89
8,anvil,89
9,flint,87



Source 2
Names with 5 or fewer characters: 17447


,Normalized Business Name,Record Count
0,foot,152
1,lynx,133
2,apex,107
3,birch,107
4,aspen,104
5,titan,103
6,cedar,101
7,ember,98
8,forge,97
9,amber,97



Source 3
Names with 5 or fewer characters: 42469


,Normalized Business Name,Record Count
0,foot,248
1,apex,220
2,sc,197
3,si,186
4,india,183
5,blue,171
6,shree,166
7,ss,165
8,st,156
9,green,151


### Country Data Quality

In [ ]:
for source_name, df in datasets.items():
    country = df["country"].astype("string").str.strip()

    missing_country = country.isna().sum()
    empty_country = country.eq("").sum()

    print(f"\n{'=' * 50}")
    print(source_name)
    print("=" * 50)

    print("Missing country values:", missing_country)
    print("Empty country values:", empty_country)
    print("\nCountry distribution:")

    display(
        country.value_counts(dropna=False)
        .rename_axis("Country")
        .reset_index(name="Record Count")
    )


Source 1
Missing country values: 0
Empty country values: 0

Country distribution:


,Country,Record Count
0,US,1323633
1,India,883188



Source 2
Missing country values: 0
Empty country values: 0

Country distribution:


,Country,Record Count
0,US,3016817
1,India,2017799



Source 3
Missing country values: 0
Empty country values: 0

Country distribution:


,Country,Record Count
0,US,3170056
1,India,2115547


## Dataset Summary

In [ ]:
summary = []

for source_name, df in datasets.items():
    summary.append({
        "Source": source_name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Total Missing Values": int(df.isna().sum().sum()),
        "Memory Usage (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

final_summary = pd.DataFrame(summary)
display(final_summary)

,Source,Rows,Columns,Total Missing Values,Memory Usage (MB)
0,Source 1,2206821,8,0,1329.52
1,Source 2,5034616,8,506908,3149.35
2,Source 3,5285603,8,527787,3273.64


In [ ]:
for source_name, df in datasets.items():
    print(f"\n{source_name}")
    print(df.isna().sum())


Source 1
entity_id                      0
business_name                  0
business_address               0
country                        0
business_name_clean            0
business_address_clean         0
business_name_normalized       0
business_address_normalized    0
dtype: int64

Source 2
entity_id                           0
business_name                       2
business_address               168967
country                             0
business_name_clean                 2
business_address_clean         168967
business_name_normalized            3
business_address_normalized    168967
dtype: int64

Source 3
entity_id                           0
business_name                      13
business_address               175916
country                             0
business_name_clean                13
business_address_clean         175916
business_name_normalized           13
business_address_normalized    175916
dtype: int64


## Business Names Associated with Multiple Addresses

In [ ]:
for source_name, df in datasets.items():
    valid = df.dropna(
        subset=["business_name_normalized", "business_address_normalized"]
    )

    address_counts = (
        valid.groupby("business_name_normalized")[
            "business_address_normalized"
        ].nunique()
    )

    multiple_addresses = (address_counts > 1).sum()

    print(
        f"{source_name}: "
        f"{multiple_addresses:,} normalized business names "
        "appear with multiple normalized addresses."
    )

Source 1: 180,757 normalized business names appear with multiple normalized addresses.
Source 2: 341,178 normalized business names appear with multiple normalized addresses.
Source 3: 354,049 normalized business names appear with multiple normalized addresses.
